In [ ]:
from openai import OpenAI
import json

In [ ]:
from recipeprep.config import get_config
config = get_config()

## RAG

In [ ]:
# Dependencies are managed in pyproject.toml.

In [ ]:
# Set OPENAI_API_KEY in your environment before running this notebook.

In [ ]:
from recipeprep.generation import get_API_response, get_generate_sys_prompt, get_recipe
from recipeprep.retrieval import (
    build_nutrient_retriever,
    build_retrievers,
    get_recipe_by_id,
    load_and_process_json,
    load_file_content,
    retrieve_food_and_nutrients,
    retrieve_similar_recipe_id,
)

In [ ]:
# Model API calls are implemented in recipeprep.generation.

## Datasources

In [ ]:
file_path_final_recipes = config.datasets_dir / "filtered_recipes_419.json"
map_file_path = config.nutrient_map_path

## Retrive the nutrients from nutrient map

In [ ]:
# Nutrient metadata is prepared by recipeprep.retrieval.

In [ ]:
retrievers = build_retrievers(file_path_final_recipes, map_file_path)
retriever_nutrient = retrievers.nutrient
retriever_recipe = retrievers.recipes

In [ ]:
# Nutrient retrieval is implemented in recipeprep.retrieval.

## Retrive similar recipe


### Prepare the RAG + Vector

In [ ]:
# Recipe document preparation is implemented in recipeprep.retrieval.

In [ ]:
# Similar-recipe lookup is implemented in recipeprep.retrieval.

In [ ]:
# Recipe lookup by ID is implemented in recipeprep.retrieval.

In [ ]:
# Both persistent vector stores were prepared above by build_retrievers().

In [ ]:
input_ingre_list = ['beef','tomato']
input_ingre_list = sorted(ingredient.lower() for ingredient in input_ingre_list)
recipe_id = retrieve_similar_recipe_id(retriever_recipe,input_ingre_list)
recipe_id

## Recipe Generation

In [ ]:
client = OpenAI()  # Reads OPENAI_API_KEY from the environment
file_path_recipe = file_path_final_recipes

In [ ]:
# Prompt construction is implemented in recipeprep.generation.

In [ ]:
# JSON loading is implemented in recipeprep.retrieval.

In [ ]:
# Recipe generation is implemented in recipeprep.generation.

#### Testing: Generated Recipe with provided example

In [ ]:
# main
print("Welcome to RecipePrep!")

avail_ingredients = ['brioche', 'rum', 'plantains', 'cider vinegar', 'brussels sprouts', 'cashews', 'fettuccine']
avail_tools = ['skillet', 'pan', 'oven']
time_input = '20'

# Generate recipe using the inputs
print("\nGenerating your recipe...\n")
recipe = get_recipe(
    client,
    avail_ingredients,
    avail_tools,
    time_input,
    temp=0.8,
    top_p=1,
    file_path=file_path_recipe,
    retriever_nutrient=retriever_nutrient,
    retriever_recipe=retriever_recipe,
    provide_example=True,
    single_prompt=False,
)
print(recipe)

## Calculate Health Score

In [ ]:
# change value with different units to gram
# the format of ingredient_dict: e.g.{'value': '3', 'unit': 'tablespoon', 'name': 'rice vinegar'}
# 3 tablespoons rice vinegar
def convert_to_grams(ingredient_dict):
  convert_table={
      'tablespoon':17.07,
      'teaspoon': 5.69,
      'ounce':28.35,
      'cup':150.00,
      'lb':453.59,
      'pound':453.59,
      'tbsp':17.07,
      'tsp':5.69,
      'oz':28.35,
      'kg':1000.00,
      'kilogram':1000.00,
      'gram': 1.00,
      'g':1.00,
      'mg': 0.001,
      'ml': 0.92, # for oil
      'clove': 4, # for garlic
  }
  unit=ingredient_dict['unit']
  value=ingredient_dict['value']
  convert_factor=convert_table.get(unit,None)
  try:
    numeric_value=eval(value)
    convert_value=numeric_value * convert_factor if convert_factor else 100
  except:
    convert_value=100
  ingredient_dict['value']=convert_value
  ingredient_dict['unit']='gram'
  return ingredient_dict


In [ ]:
import re

def get_health_score_with_rag(retriever,recipe):
  ingredients=recipe.get("processed_ingredients")
  pure_ingredients=recipe.get("pure_ingredients")
  nutrient_map={
      "Protein":0,
      "Carbohydrate":0,
      "Sugars, total":0,
      "Sodium, Na":0,
      "Total Fat":0,
      "Fatty acids, saturated, total":0,
      "Fibre, total dietary": 0,
      "Energy (kJ)": 0,
  }

  # print(ingredients)

  for i,ingredient in enumerate(ingredients):
    match = re.match(r"([\d./]+)\s*([a-zA-Z]+)?\s*(.*)", ingredient)
    if match:
      value = match.group(1).strip()
      unit = match.group(2) if match.group(2) else ""
      if len(pure_ingredients)==len(ingredients):
        name = pure_ingredients[i]
      else:
        name = match.group(3).strip()


      if unit.endswith("s"):  # Handle plural forms
        unit = unit[:-1]
      parsed_ingredient={"value": value, "unit": unit, "name": name}
      # print(parsed_ingredient)
      ingredient_dict=convert_to_grams(parsed_ingredient)
      print(ingredient_dict)
      matched_ingredient,nutrients=retrieve_food_and_nutrients(retriever,ingredient_dict["name"])
      # print(matched_ingredient)
      # print(nutrients)
      nutrient_pattern = r"'value': ([\d.]+), 'nutrient_name': '([^']+)'"
      matches=re.findall(nutrient_pattern,nutrients)
      for value,name in matches:
        if name in nutrient_map:
          nutrient_map[name]+=float(value)*ingredient_dict["value"]/100
        # if name=="Sodium, Na":
        #   print(value, ingredient_dict["value"], float(value)*ingredient_dict["value"]/100)

  health_score=0
  score_summary={
      "Protein": 0,
      "Carbohydrate": 0,
      "Sugars": 0,
      "Sodium": 0,
      "Fats": 0,
      "Saturated Fats": 0,
      "Fibers": 0
  }
  protein_energy=nutrient_map['Protein']*17
  carbo_energy=nutrient_map['Carbohydrate']*17
  fat_energy=nutrient_map['Total Fat']*37
  sugar_energy=nutrient_map['Sugars, total']*17
  sat_fat_energy=nutrient_map['Fatty acids, saturated, total']*37
  fiber_energy=nutrient_map['Fibre, total dietary']*8
  sodium_energy=nutrient_map['Sodium, Na']*0
  total_energy=nutrient_map['Energy (kJ)']

  print(nutrient_map)

  # Define thresholds
  requirements = {
      "Protein": {"min": total_energy * 0.1, "max": total_energy * 0.35, "value": protein_energy, "reason": "Protein not enough" if protein_energy < total_energy * 0.1 else "Too much protein"},
      "Carbohydrate": {"min": total_energy * 0.45, "max": total_energy * 0.65, "value": carbo_energy, "reason": "Carbohydrates not enough" if carbo_energy < total_energy * 0.55 else "Too many carbohydrates"},
      "Sugars": {"max": total_energy * 0.1, "value": sugar_energy, "reason": "Too much sugar"},
      "Sodium": {"max": 500000, "value": nutrient_map['Sodium, Na'], "reason": "Too much sodium"},
      "Fats": {"min": total_energy * 0.15, "max": total_energy * 0.3, "value": fat_energy, "reason": "Fats not enough" if fat_energy < total_energy * 0.15 else "Too many fats"},
      "Saturated Fats": {"max": total_energy * 0.1, "value": sat_fat_energy, "reason": "Too much saturated fat"},
      "Fibers": {"min": 6, "value": nutrient_map['Fibre, total dietary'], "reason": "Fibers not enough"}
  }

  # Initialize health score and summary
  health_score = 0
  score_summary = {}

  # Evaluate each nutrient
  for nutrient, thresholds in requirements.items():
      value = thresholds["value"]
      min_val = thresholds.get("min", float("-inf"))  # Default min to negative infinity
      max_val = thresholds.get("max", float("inf"))  # Default max to positive infinity

      if min_val <= value <= max_val:
          health_score += 1
          score_summary[nutrient] = 1
      else:
          print(f"Failed: {nutrient}\nReason: {thresholds['reason']} (Value: {value})\n")

  # Final health score and score summary
  print(f"Health Score: {health_score}")
  print(f"Score Summary: {score_summary}")

  return health_score, score_summary

In [ ]:
recipe_health = json.loads(recipe)
print(get_health_score_with_rag(retriever_nutrient,recipe_health))

## Calculate Relevancy Score

In [ ]:
#recipe = json.loads(recipe)
# Legacy shell invocation removed; call the imported evaluation function directly.

In [ ]:
# nutrient_map_path = config.nutrient_map_path
# focused_tools = ["stove", "oven"]
nutrient_map_path = 'ingredient_nutrient_map.json'
focused_tools = ["pan"]

In [ ]:
def check_cooking_tools(input_tools,recipe,focused_tools):
  recipe_tools=recipe.get("required_tools")
  for tool in focused_tools:
    if tool in recipe_tools and tool not in input_tools:
      return False
  return True


In [ ]:
def check_cooking_time(input_time,recipe):
  # cooking_time_str=recipe.get("cooking_time")
  # cooking_time = int(''.join(filter(str.isdigit, cooking_time_str)))
  cooking_time = recipe.get("cooking_time")
  if cooking_time <= input_time:
    print("Meet cooking time requirements")
    return 0

  print(f"Does not meet cooking time requirements, time difference is: {cooking_time - input_time}")
  return cooking_time - input_time

In [ ]:
# Shared nutrient vector-store functions are imported from recipeprep.retrieval.

In [ ]:
def get_matched_list(ingredient_list,retriever):
    matched_ingredient_list=[]
    for ingredient in ingredient_list:
      matched_ingredient,nutrients=retrieve_food_and_nutrients(retriever,ingredient)
      matched_ingredient_list.append(matched_ingredient)
    return matched_ingredient_list

def compare_ingredient_list(user_root_list,recipe_root_list):
  user_set={ing for ing in user_root_list}
  recipe_set={ing for ing in recipe_root_list}
  is_covered=recipe_set.issubset(user_set)
  common_ingredients=user_set&recipe_set
  overlap_rate=(len(common_ingredients)/len(recipe_set))*100
  return is_covered,overlap_rate

def get_similarity(input_ingredients,recipe,retriever):
  # recipe_ingredients=recipe.get("pure_ingredients")
  # recipe_ingredients_list = [ingredient.strip() for ingredient in recipe_ingredients.split("\n")]
  recipe_ingredients_list = recipe.get("pure_ingredients")
  matched_ingredient_list_1=get_matched_list(input_ingredients,retriever)
  matched_ingredient_list_2=get_matched_list(recipe_ingredients_list,retriever)
  is_covered,overlap_rate=compare_ingredient_list(matched_ingredient_list_1,matched_ingredient_list_2)
  return is_covered,overlap_rate

In [ ]:
def relevance_evaluation(output_used_tools, input_tools, input_time, input_ingredients, recipe, nutrient_map_path):
  # os.environ["OPENAI_API_KEY"]=getpass.getpass()
  retriever = build_nutrient_retriever(nutrient_map_path)

  hard_constraint=check_cooking_tools(input_tools, recipe, output_used_tools)
  cooking_time_constraint=check_cooking_time(input_time,recipe)
  is_covered, overlap_rate=get_similarity(input_ingredients,recipe,retriever)

  relevance_eval={
      'cooking_tools': hard_constraint,
      'cooking_time': cooking_time_constraint,
      'ingredient_overlap_rate': overlap_rate
  }
  return relevance_eval

In [ ]:
recipe_relevant = json.loads(recipe)
print(relevance_evaluation(focused_tools, avail_tools, int(time_input), avail_ingredients, recipe_relevant, nutrient_map_path))

## Testing Scripts

In [ ]:
focused_tools = ["stove", "oven", "skillet", "pan"]

In [ ]:
output_recipe = json.loads(recipe)
health_score, score_summary = get_health_score_with_rag(retriever_nutrient, output_recipe)

print(relevance_evaluation(focused_tools, avail_tools, int(time_input), avail_ingredients, output_recipe, map_file_path))

## User Interface

In [ ]:
# The Gradio dependency is provided by the project app extra.

In [ ]:
import gradio as gr
import json
import textwrap

In [ ]:
import gradio as gr

# Wrapper function for Gradio to call `get_recipe`
def generate_recipe(ingredients, tools, time, provide_example, single_prompt):
    try:
        # Split comma-separated inputs into lists
        ingredients = str(ingredients)
        tools = str(tools)
        ingredients_list = [ingredient.strip() for ingredient in ingredients.split(",")]
        tools_list = [tool.strip() for tool in tools.split(",")]

        # Mock client initialization (replace with your actual client setup)
        client = OpenAI()  # Reads OPENAI_API_KEY from the environment

        # Call the `get_recipe` function
        recipe = get_recipe(
            client=client,
            ingredients=ingredients_list,
            tools=tools_list,
            time=str(time),
            temp=0.8,
            top_p=1,
            file_path=config.datasets_dir / "filtered_recipes_419.json",
            retriever_nutrient=retriever_nutrient,
            retriever_recipe=retriever_recipe,
            provide_example=provide_example,
            single_prompt=single_prompt,
        )
        return recipe
    except Exception as e:
        return f"Error: {str(e)}"

# Define Gradio inputs
ingredients_input = gr.Textbox(label="Enter your ingredients", placeholder="Enter ingredients")
tools_input = gr.Textbox(label="Enter your cooking requirements", placeholder="Enter tools")
time_input = gr.Number(label="Enter your preferred cooking Time (in min)")
provide_example_toggle = gr.Checkbox(label="Provide Example Recipes", value=True)
single_prompt_toggle = gr.Checkbox(label="Use Single Prompt Format", value=False)

# Gradio interface
interface = gr.Interface(
    fn=generate_recipe,
    inputs=[
        ingredients_input,
        tools_input,
        time_input,
        provide_example_toggle,
        single_prompt_toggle,
    ],
    outputs="text",
    title="Recipe Generator",
    description="Generate personalized recipes based on your ingredients, tools, and preferences.",
)

# Launch the Gradio interface
interface.launch(debug=True)
